In [0]:
select status, count(*) as row_status
from shopstream.core.bronze_orders
group by status
order by row_status desc;


In [0]:
select(
    select count(*) from
    (
        select order_line_id from shopstream.core.bronze_orders
        group by order_line_id having count(*) > 1
    )
) as duplicated_line_ids,
(select count(*) from shopstream.core.bronze_orders
where quantity < 0) as bad_quality_rows;

In [0]:
create or replace table shopstream.core.silver_orders as
with deduped as(
    select *,
    row_number() over (partition by order_line_id order by order_ts) as rn
    from shopstream.core.bronze_orders
)
select order_line_id,
       order_id,
       customer_id,
       product_id,
       cast(quantity as int) as quantity,
       cast(unit_price as double) as unit_price,
       round(quantity * unit_price, 2) as line_revenue,
       cast(order_ts as timestamp) as order_ts,
       lower(status) as status,
       nullif(coupon_code, '') as coupon_code
from deduped
where rn = 1
  and quantity > 0;
select count(*) as silver_rows from shopstream.core.silver_orders;

In [0]:

CREATE OR REPLACE TABLE shopstream.core.silver_customers AS
SELECT
  customer_id,
  name,
  email,
  city,
  country,
  CAST(signup_date AS DATE) AS signup_date,
  signup_channel
FROM shopstream.core.bronze_customers;

CREATE OR REPLACE TABLE shopstream.core.silver_products AS
SELECT
  product_id,
  product_name,
  category,
  CAST(unit_price AS DOUBLE) AS unit_price,
  CAST(unit_cost AS DOUBLE) AS unit_cost,
  ROUND(unit_price - unit_cost, 2) AS unit_margin
FROM shopstream.core.bronze_products;



In [0]:
describe history shopstream.core.silver_orders

In [0]:
delete from shopstream.core.silver_orders where status = 'cancelled';
select
(select count(*) from shopstream.core.silver_orders) as after_change,
(select count(*) from shopstream.core.silver_orders version as of 0) as before_change;